# ingest circuit csv file
- read file using spark dataframe reader api
- add metadata columns
 > source file and
 > ingestion timestamp
- wrrite to bronze delta table 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

race_schema = StructType([
        StructField('season',   IntegerType()),
        StructField('round',    IntegerType()),
        StructField('url',      StringType()),
        StructField('racename', StringType()),
        StructField('date',     DateType()),
        StructField('circuitId', StringType())
])

In [0]:
races_df =(
    spark.read.format('csv')
    .option('header', True)
    .schema(race_schema)
    .load('/Volumes/formular1/landing/files/races.csv')
)

In [0]:
display(races_df)

### step 2 metadata

In [0]:
from pyspark.sql import functions as F
races_final_df = (races_df
                  .withColumn('ingestion_timestamp', F.current_timestamp())
                  .withColumn('source_file', F.col('_metadata.file_path'))

)

In [0]:
display(races_final_df)

write files to delta format

In [0]:
(
    races_final_df.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('formular1.bronze.races')
)